# YouTube Title Predictor

**Problem:** Small travel channels (like my channel KarmicSteps) have to pick one title
from several ideas. This project predicts whether a long-form travel video will beat its
channel's recent typical views, so candidate titles can be ranked before publishing.

**Data:** 21,814 videos from 70 small English-language travel channels (1k–100k subscribers),
collected with the YouTube Data API on 2026-10-26 (see `collect_data.py`).

## 1. Data cleaning


In [2]:
import pandas as pd
import numpy as np

### 1.1 Load the data and fix the types

In [3]:
COLLECTION_DATE = pd.Timestamp("2026-10-26", tz="UTC")

videos = pd.read_csv("data/videos.csv")
print(videos.shape)
videos.head()

(21814, 9)


,video_id,channel_id,title,published_at,duration,view_count,tags,description,was_live
0,7uNx8yEasm4,UCYAIbGWcLdohnpRh-YLXzRw,Kuala Lumpur might be Asia’s most convenient city,2026-10-24T11:01:11Z,PT17M6S,20404,kuala lumpur|Kuala Lumpur vlog|Kuala Lumpur Ma...,Get 5% off your own tomtoc bag using code *SAR...,False
1,GkcnSulE3SU,UCYAIbGWcLdohnpRh-YLXzRw,This INCREDIBLE city is only 1 hour from KUALA...,2026-10-21T11:01:01Z,PT29M40S,41837,Seremban Malaysia|Seremban travel|Seremban vlo...,We left Melaka behind and took the local buses...,False
2,Vp-tGcRs2EQ,UCYAIbGWcLdohnpRh-YLXzRw,You need to know this if you're heading to Asi...,2026-10-21T07:31:21Z,PT45S,2867,travel tips|travelling asia|southeast asia|bac...,"👉 Before we dive in, comment SLYNUMBER to get ...",False
3,K9nwiOuEIOw,UCYAIbGWcLdohnpRh-YLXzRw,What's on Asia's most mysterious island? 🤔 (Fu...,2026-10-21T04:09:38Z,PT34S,13408,pulau besar|pulau besar melaka|pulau besar mal...,"Just off the coast of Melaka is Pulau Besar, a...",False
4,sSOKZa5xOCY,UCYAIbGWcLdohnpRh-YLXzRw,European countries could never... (Full video ...,2026-10-20T07:33:27Z,PT30S,14537,melaka|malacca|melaka malaysia|malacca malaysi...,We've been in Melaka for a few days and now it...,False


In [4]:
videos["tags"] = videos["tags"].fillna("")
videos["description"] = videos["description"].fillna("")
videos["published_at"] = pd.to_datetime(videos["published_at"])
videos["duration_s"] = videos["duration"].apply(lambda d: pd.Timedelta(d).total_seconds())
videos["age_days"] = (COLLECTION_DATE - videos["published_at"]).dt.days
videos.dtypes

video_id                        str
channel_id                      str
title                           str
published_at    datetime64[us, UTC]
duration                        str
view_count                    int64
tags                            str
description                     str
was_live                       bool
duration_s                  float64
age_days                      int64
dtype: object

### 1.2 Filter: long-form only, no live streams

In [8]:
# Filter videos
# Keep long-form only: duration_s > 180
# Drop past live streams: was_live is False 
videos = videos[videos['duration_s'] > 180]
print("After filtering long-form videos:", len(videos))
videos = videos[~videos["was_live"]]
print("After removing live streams:", len(videos))

After filtering long-form videos: 10047
After removing live streams: 10047


## 2. Target: beat the channel's recent median

Each video is compared with the median views of its own channel's previous 20 long-form videos. This removes the effect of channel size, and use only information available at upload time.

In [9]:
videos = videos.sort_values(["channel_id", "published_at"])         # channel first, then publish date

videos["baseline"] = (
    videos.groupby("channel_id")["view_count"]
    .transform(lambda s: s.shift(1).rolling(20, min_periods=20).median())
)

videos["beat_baseline"] = (videos["view_count"] > videos["baseline"]).astype(int)

print("Rows with a baseline:", videos["baseline"].notna().sum())
videos[["channel_id", "published_at", "view_count", "baseline", "beat_baseline"]].head(25)


Rows with a baseline: 8696


,channel_id,published_at,view_count,baseline,beat_baseline
16372,UC-9wCcxlOzrejf8ibN5blPg,2022-04-26 12:00:37+00:00,200,NaN,0
16371,UC-9wCcxlOzrejf8ibN5blPg,2022-05-26 11:00:04+00:00,249,NaN,0
16369,UC-9wCcxlOzrejf8ibN5blPg,2022-06-16 05:00:07+00:00,39528,NaN,0
16368,UC-9wCcxlOzrejf8ibN5blPg,2022-06-23 05:00:07+00:00,2077,NaN,0
16367,UC-9wCcxlOzrejf8ibN5blPg,2022-06-30 05:00:19+00:00,4754,NaN,0
16366,UC-9wCcxlOzrejf8ibN5blPg,2022-07-07 05:00:09+00:00,5565,NaN,0
16365,UC-9wCcxlOzrejf8ibN5blPg,2022-07-15 05:16:48+00:00,18524,NaN,0
16364,UC-9wCcxlOzrejf8ibN5blPg,2022-07-22 05:29:14+00:00,4725,NaN,0
16363,UC-9wCcxlOzrejf8ibN5blPg,2022-07-26 05:33:48+00:00,996,NaN,0
16362,UC-9wCcxlOzrejf8ibN5blPg,2022-07-29 05:25:50+00:00,89089,NaN,0


### 2.2 Drop rows without a baseline, apply the age window:

In [10]:
df = videos[videos["baseline"].notna()]
print("With a baseline:", len(df))

df = df[(df["age_days"] >= 30) & (df["age_days"] <= (3 * 365))]
print("With age between 30 days and 3 years:", len(df))

print("Channels:", df["channel_id"].nunique())
df["beat_baseline"].value_counts(normalize=True)

With a baseline: 8696
With age between 30 days and 3 years: 3749
Channels: 55


beat_baseline
1    0.503067
0    0.496933
Name: proportion, dtype: float64

### 2.2 Drop rows without a baseline, apply the age window


In [11]:
per_channel = df["channel_id"].value_counts()
print(per_channel.describe())
print("Channels with < 20 videos:", (per_channel < 20).sum())

count     55.000000
mean      68.163636
std       73.194506
min        1.000000
25%       11.000000
50%       43.000000
75%       92.500000
max      287.000000
Name: count, dtype: float64
Channels with < 20 videos: 20


In [ ]:
def ascii_share(title):
    letters = [c for c in title if c.isalpha()]
    if not letters:
        return 1.0
    return sum(c.isascii() for c in letters) / len(letters)



## 3. Exploratory data analysis
### 3.1 Target balance
### 3.2 Videos per channel
### 3.3 Numeric features (duration, publish time, tags, description length)
### 3.4 Title words: most common words for beat vs not beat

## 4. Feature engineering
### 4.1 Title stats
### 4.2 TF-IDF on titles

## 5. Models (GroupKFold by channel)
### 5.1 Logistic regression without title text (baseline)
### 5.2 Logistic regression with TF-IDF
### 5.3 Random forest
### 5.4 XGBoost
### 5.5 Comparison and final choice

## 6. KarmicSteps sanity check